# Credit Card Customer Segmentation
Segmenting 8,950 cardholders by behaviour with K-Means, then turning the clusters into retention strategy. This notebook mirrors `src/analysis.py` with narration.

## 1. Load & clean
Drop the ID, median-impute missing values.

In [1]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv('../data/raw/credit_card_customers.csv').drop(columns=['CUST_ID'])
print('shape:', df.shape, '| missing:', int(df.isna().sum().sum()))
df = df.fillna(df.median(numeric_only=True))
df.head()

shape: (8950, 17) | missing: 314


,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,312.343947,0.000000,12
4,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


## 2. Feature engineering
Log-transform skewed monetary fields, then standardise.

In [2]:
monetary = ['BALANCE','PURCHASES','ONEOFF_PURCHASES','INSTALLMENTS_PURCHASES',
            'CASH_ADVANCE','CREDIT_LIMIT','PAYMENTS','MINIMUM_PAYMENTS']
X = df.copy()
for c in monetary: X[c] = np.log1p(X[c].clip(lower=0))
Xs = StandardScaler().fit_transform(X)
Xs.shape

(8950, 17)

## 3. Choose k
Sweep k = 2–8 and score with the silhouette coefficient.

In [3]:
for k in range(2,9):
    lab = KMeans(k, random_state=42, n_init=10).fit_predict(Xs)
    print(k, round(silhouette_score(Xs, lab),3))

2 0.221


3 0.21


4 0.205


5 0.195


6 0.205


7 0.195


8 0.197


The score is modest and flat — typical for overlapping behavioural data. We pick **k = 3** for three distinct, actionable personas.

In [4]:
km = KMeans(3, random_state=42, n_init=20)
labels = km.fit_predict(Xs)
df['cluster'] = labels
print('silhouette (k=3):', round(silhouette_score(Xs, labels),3))

silhouette (k=3): 0.21


## 4. Profile the segments
Averages on the original (un-logged) scale.

In [5]:
prof = df.groupby('cluster').agg(size=('BALANCE','size'),
    balance=('BALANCE','mean'), purchases=('PURCHASES','mean'),
    cash_advance=('CASH_ADVANCE','mean'), prc_full=('PRC_FULL_PAYMENT','mean'),
    purch_freq=('PURCHASES_FREQUENCY','mean')).round(2)
prof

,size,balance,purchases,cash_advance,prc_full,purch_freq
cluster,,,,,,
0,2847,195.87,463.49,55.19,0.27,0.53
1,3043,2018.75,2373.43,700.57,0.17,0.85
2,3060,2386.06,142.74,2115.02,0.03,0.09


## 5. The 12-month tenure cliff

In [6]:
(df['TENURE'].value_counts(normalize=True).sort_index()*100).round(1)

TENURE
6      2.3
7      2.1
8      2.2
9      2.0
10     2.6
11     4.1
12    84.7
Name: proportion, dtype: float64

## 6. Reproduce all outputs
The full pipeline (figures + dashboard JSON) lives in `src/analysis.py`.

In [7]:
import subprocess, sys
print(subprocess.run([sys.executable,'../src/analysis.py'], capture_output=True, text=True).stdout)

n=8950  k=3  silhouette=0.210  PCA2D var=0.52
                   segment  size  share  ...  cash_advance  prc_full_payment  tenure
cluster                                  ...                                        
0        Marginal Spenders  2847   31.8  ...         55.19              0.27   11.33
1        Power Transactors  3043   34.0  ...        700.57              0.17   11.82
2        Revolving Debtors  3060   34.2  ...       2115.02              0.03   11.39

[3 rows x 8 columns]

